# RAG の回答の正しさを LLM に判定させる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/24_rag/rag_eval_llm_judge.ipynb)

NLP クックブック 24-11 ／ RAG（検索拡張生成）

正解の文字列を含むかどうかの判定では、言い換えた正答を誤りにしてしまう。回答が正解と同じ内容かを、LLM に判定させたい。

In [1]:
import numpy as np
import pandas as pd
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま)
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
llm = AutoModelForCausalLM.from_pretrained(name).eval()
YES_NO = [tokenizer(c, add_special_tokens=False)["input_ids"][0] for c in ["はい", "いいえ"]]

@torch.no_grad()
def p_yes(prompt):
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, tokenize=False)
    ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"]
    return float(llm(ids).logits[0, -1, YES_NO].softmax(-1)[0])

def judge_correct(question, gold, answer):
    return p_yes("次の質問に対する「回答」が、「正解」と同じ内容を答えているかを判定してください。表現が違っても内容が同じなら「はい」、"
                 "内容が違う・答えていない場合は「いいえ」と、一語だけで答えてください。\n\n"
                 f"質問: {question}\n正解: {gold}\n回答: {answer}")

# (質問, 正解, システムの回答, 人が判定した正誤)
data = [
    ("富士山の標高は？", "3776 メートル", "富士山の標高は 3776 メートルです。", True),
    ("富士山の標高は？", "3776 メートル", "3,776m です。", True),
    ("富士山の標高は？", "3776 メートル", "約 3800 メートルです。", False),
    ("日本で一番大きい湖は？", "琵琶湖", "滋賀県にある琵琶湖です。", True),
    ("日本で一番大きい湖は？", "琵琶湖", "霞ヶ浦です。", False),
    ("日本で一番大きい湖は？", "琵琶湖", "琵琶湖ではなく、霞ヶ浦です。", False),
    ("『吾輩は猫である』の作者は？", "夏目漱石", "漱石です。", True),
    ("『吾輩は猫である』の作者は？", "夏目漱石", "わかりません。", False),
    ("『吾輩は猫である』の作者は？", "夏目漱石", "明治時代の小説家が書きました。", False),
    ("第二次世界大戦が終わったのは何年？", "1945 年", "昭和 20 年です。", True),
    ("第二次世界大戦が終わったのは何年？", "1945 年", "1945 年 8 月に終わりました。", True),
    ("第二次世界大戦が終わったのは何年？", "1945 年", "1939 年に始まりました。", False),
]
import re, unicodedata
norm = lambda s: re.sub(r"[\s、。,]", "", unicodedata.normalize("NFKC", s))
rows = [{"回答": a, "人": h, "文字列照合": norm(g) in norm(a), "LLM (はいの確率)": round(judge_correct(q, g, a), 2)} for q, g, a, h in data]
df = pd.DataFrame(rows)
df["LLM"] = df["LLM (はいの確率)"] > 0.5
df

,回答,人,文字列照合,LLM (はいの確率),LLM
0,富士山の標高は 3776 メートルです。,True,True,1.00,True
1,"3,776m です。",True,False,1.00,True
2,約 3800 メートルです。,False,False,1.00,True
3,滋賀県にある琵琶湖です。,True,True,1.00,True
4,霞ヶ浦です。,False,False,0.00,False
5,琵琶湖ではなく、霞ヶ浦です。,False,True,0.00,False
6,漱石です。,True,False,1.00,True
7,わかりません。,False,False,0.00,False
8,明治時代の小説家が書きました。,False,False,0.97,True
9,昭和 20 年です。,True,False,1.00,True


In [2]:
print(f"人の判定との一致率: 文字列照合 {(df['文字列照合'] == df['人']).mean():.2f} / LLM {(df['LLM'] == df['人']).mean():.2f}")
print("文字列照合だけが間違えた例:", df[(df["文字列照合"] != df["人"]) & (df["LLM"] == df["人"])]["回答"].tolist())
print("LLM だけが間違えた例:", df[(df["LLM"] != df["人"]) & (df["文字列照合"] == df["人"])]["回答"].tolist())

人の判定との一致率: 文字列照合 0.67 / LLM 0.83
文字列照合だけが間違えた例: ['3,776m です。', '琵琶湖ではなく、霞ヶ浦です。', '漱石です。', '昭和 20 年です。']
LLM だけが間違えた例: ['約 3800 メートルです。', '明治時代の小説家が書きました。']


In [3]:
# 同じ方法で、回答が質問に答えているか (関連性) と、資料に基づいているか (忠実性) も判定できる
def judge_relevant(question, answer):
    return p_yes("次の「回答」は、「質問」に対する答えになっていますか (内容が正しいかどうかは問いません)。答えになっていれば「はい」、"
                 f"質問と関係のないことを述べていれば「いいえ」と、一語で答えてください。\n\n質問: {question}\n回答: {answer}")

def judge_faithful(context, answer):
    return p_yes("資料だけを根拠にして、次の「主張」が正しいと言えるかを判定してください。資料から正しいと分かるなら「はい」、"
                 f"資料と食い違う、または資料からは分からない内容を含むなら「いいえ」と、一語で答えてください。\n\n資料: {context}\n主張: {answer}")

context = "クックブック食堂の営業時間は平日 11 時から 15 時、土日は 11 時から 20 時です。月曜日は定休日です。"
q = "日曜日は何時まで開いていますか？"
for a in ["日曜日は 20 時まで営業しています。", "定休日は月曜日です。", "20 時までで、ラストオーダーは 19 時半です。", "日曜日は 15 時までです。"]:
    print(f"関連性 {judge_relevant(q, a):.2f} / 忠実性 {judge_faithful(context, a):.2f} | {a}")

関連性 0.73 / 忠実性 1.00 | 日曜日は 20 時まで営業しています。
関連性 0.00 / 忠実性 1.00 | 定休日は月曜日です。
関連性 0.71 / 忠実性 0.00 | 20 時までで、ラストオーダーは 19 時半です。
関連性 1.00 / 忠実性 0.00 | 日曜日は 15 時までです。


- 文字列照合は、表記の違い (3,776m)、言い換え（昭和 20 年）、略した答え（漱石）を誤りにする。逆に、正解の語を含むだけの誤答（「琵琶湖ではなく…」）を正答にする。LLM による判定はこの両方に対処できる。
- 「はい」「いいえ」の確率を直接読むと、1 回の計算で済み、あいまいな出力の解釈も要らない。確率が 0.5 付近のものは人が確認する、という運用ができる。
- 判定の質はプロンプトの書き方に大きく左右される。「すべて資料に書かれていますか」と聞くと正しい回答まで「いいえ」になり、「資料だけを根拠に正しいと言えるか」と聞くと安定した、というように、言い回しで結果が変わる。
- 判定役の LLM も間違える。人の判定を付けた数十〜百件で一致率を確かめてから使う（ここの 12 件は動作確認用）。
- 正しさ・関連性・忠実性を LLM で採点する考え方をまとめたライブラリに RAGAS がある。API のモデルを使う前提だが、指標の定義は参考になる。
- 小さなモデルは判定の精度が限られる。最終的な評価には、より大きなモデル（LLM の評価と分析の章の LLM-as-a-judge を参照）か人手を使う。